In [2]:
from google.colab import files
from transformers import GPT2Tokenizer
import os

from google.colab import files

uploaded = files.upload()

def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        lines = file.readlines()
    prompts = []
    labels = []
    for line in lines:
        if "=>" in line:
            parts = line.split("=>")
            if len(parts) == 2:
                prompts.append(parts[0].strip())
                labels.append(parts[1].strip())
    return prompts, labels

train_texts, train_labels = load_data("fine_tune_dataset1.txt")

print("Nombre d'exemples :", len(train_texts))
print("Premier exemple :", train_texts[0], "=>", train_labels[0])


FileNotFoundError: [Errno 2] No such file or directory: 'fine_tune_dataset1.txt'

In [118]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

tokenizer.pad_token = tokenizer.eos_token  # Utiliser le token EOS comme token de padding

print("Pad token ajouté :", tokenizer.pad_token)
print(train_texts[:3])



Pad token ajouté : <|endoftext|>
["Classify the following statement: 'I had a wonderful time with my friends today.' Is it positive or negative?", "Classify the following statement: 'That meal was absolutely delicious.' Is it positive or negative?", "Classify the following statement: 'I was so tired I couldn't focus on anything.' Is it positive or negative?"]


In [119]:
def tokenize_and_create_labels(texts):
    encodings = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors='pt')

    input_ids = encodings['input_ids']
    labels = input_ids.clone()
    labels[:, :-1] = input_ids[:, 1:].clone()
    labels[:, -1] = tokenizer.pad_token_id

    encodings['labels'] = labels
    return encodings

train_encodings = tokenize_and_create_labels(train_texts)

print("Shape des inputs d'entraînement :", train_encodings['input_ids'].shape)
print("Shape des labels d'entraînement :", train_encodings['labels'].shape)


Shape des inputs d'entraînement : torch.Size([331, 27])
Shape des labels d'entraînement : torch.Size([331, 27])


In [120]:
from datasets import Dataset

train_dataset = Dataset.from_dict(train_encodings)

print(train_dataset[0])




{'input_ids': [9487, 1958, 262, 1708, 2643, 25, 705, 40, 550, 257, 7932, 640, 351, 616, 2460, 1909, 2637, 1148, 340, 3967, 393, 4633, 30, 50256, 50256, 50256, 50256], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0], 'labels': [1958, 262, 1708, 2643, 25, 705, 40, 550, 257, 7932, 640, 351, 616, 2460, 1909, 2637, 1148, 340, 3967, 393, 4633, 30, 50256, 50256, 50256, 50256, 50256]}


In [121]:
from transformers import GPT2LMHeadModel, Trainer, TrainingArguments

model = GPT2LMHeadModel.from_pretrained("gpt2")

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    report_to=None
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

trainer.train()


Step,Training Loss
10,8.777900
20,8.342500
30,7.920000
40,6.984000
50,6.239500
60,5.329600
70,4.554000
80,3.714400
90,2.904500
100,2.191400


TrainOutput(global_step=249, training_loss=2.881901336961003, metrics={'train_runtime': 811.1065, 'train_samples_per_second': 1.224, 'train_steps_per_second': 0.307, 'total_flos': 13682618496000.0, 'train_loss': 2.881901336961003, 'epoch': 3.0})

In [122]:
model.save_pretrained("fine_tuned_gpt_finall")
tokenizer.save_pretrained("fine_tuned_gpt_finall")

model = GPT2LMHeadModel.from_pretrained("fine_tuned_gpt_finall")
tokenizer = GPT2Tokenizer.from_pretrained("fine_tuned_gpt_finall")



In [1]:
prompt = "Classify the following statement: 'I love the feeling of a new book in my hands.' Is it positive or negative?"

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    inputs["input_ids"],
    max_length=100,
    num_return_sequences=1,
    no_repeat_ngram_size=2,
    top_k=50,
    top_p=0.95,
    temperature=0.7
)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generated_text)


NameError: name 'tokenizer' is not defined